# Model 1: Inference Only

**Goal:** Send all 644 Austrian tax law questions to GPT-4o-mini and save the answers.

This is the simplest model — we use a pre-trained model (GPT-4o-mini) directly via the OpenAI API, without any training.

In [ ]:
# Install required packages (run once)
!pip install openai pandas

In [ ]:
import pandas as pd
from openai import OpenAI

# ⚠ Replace with your API key — do not commit this to GitHub!
API_KEY = "INSERT-YOUR-KEY-HERE"
client = OpenAI(api_key=API_KEY)

In [ ]:
# Load the dataset (644 Austrian tax law questions)
df = pd.read_csv("../../dataset_clean.csv")
print(f"Loaded {len(df)} questions")
df.head(3)

In [ ]:
# System prompt: tells GPT how to answer
# Short, precise, German, with law references (§ numbers) — same style as correct answers
system_prompt = """Du bist ein österreichischer Steuerrechtsexperte.
Beantworte die Frage präzise und nenne die relevanten Gesetzesstellen (z.B. § 7 Abs. 1 KStG).
Antworte auf Deutsch in 1–3 kurzen Sätzen."""

In [ ]:
# Run inference: ask GPT each of the 644 questions
answers = []

for i, row in df.iterrows():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": row["prompt"]}
        ],
        max_completion_tokens=300,
        temperature=0
    )
    answer = response.choices[0].message.content.strip()
    answers.append(answer)
    print(f"{i+1}/{len(df)} | {row['id']} | {answer[:70]}...")

In [ ]:
# Save results as CSV
results = pd.DataFrame({"id": df["id"], "answer": answers})
results.to_csv("../results/model1_results.csv", index=False)
print(f"Saved {len(results)} answers to model1_results.csv")
results.head(3)